# Seed study: how much do the scores vary between training runs?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/skumagai3/deepvoid-demo/blob/main/notebooks/seed_study.ipynb)

This notebook is optional. You don't need it to use the demo; the README already shows its result
(`assets/seed_study.png`). It answers one question: are the differences between models, especially
curriculum vs. training from scratch, bigger than the run-to-run noise?

It retrains the three models several times with the published settings (min-max normalization,
overlapping training subcubes) and the same training/validation split, changing only the training
seed (weight initialization and shuffling). The default is 5 seeds, each of which retrains all three
models, so use a GPU runtime. The last cell downloads `seed_study_outputs.zip`.

In [ ]:
import os, subprocess
os.environ["GIT_TERMINAL_PROMPT"] = "0"     # fail fast instead of asking for a password
if not os.path.isdir("dvdemo"):
    if os.path.isdir("../dvdemo"):
        os.chdir("..")
    elif subprocess.run(["git", "clone", "-q", "--depth", "1",
                         "https://github.com/skumagai3/deepvoid-demo.git"]).returncode == 0:
        os.chdir("deepvoid-demo")
    else:  # repo not public yet: upload the zip of the repo instead
        from google.colab import files
        print("Upload deepvoid-demo.zip")
        name = next(iter(files.upload()))
        subprocess.run(["unzip", "-q", "-o", name], check=True)
        os.chdir("deepvoid-demo")
print("working in", os.getcwd())

In [ ]:
import json, os, time
import numpy as np
import tensorflow as tf
from dvdemo.io import load_sample
from dvdemo.train import ArchConfig, StageConfig, run_all

print("TensorFlow", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))
sample = load_sample("data/tng300_128.npz")

In [ ]:
N_SEEDS = 5
common = dict(max_epochs=300, batch_size=8, normalization="minmax", overlap=True, verbose=0)
runs = []
for k in range(N_SEEDS):
    t = time.time()
    print(f"=== seed {k} ===", flush=True)
    res = run_all(sample, out_dir=f"runs_seeds/seed{k}/weights", results_dir=f"runs_seeds/seed{k}/results",
                  arch=ArchConfig(),
                  dense=StageConfig(learning_rate=1e-3, seed=100 * k, **common),
                  curriculum=StageConfig(learning_rate=1e-4, frozen_blocks=1, seed=100 * k + 1, **common),
                  scratch=StageConfig(learning_rate=1e-3, seed=100 * k + 2, **common))
    runs.append({name: {m: r["val_metrics"][m] for m in ("void_f1", "mcc", "balanced_accuracy", "halo_f1")}
                 for name, r in res.items()})
    print(f"    {(time.time() - t) / 60:.1f} min", flush=True)

os.makedirs("results", exist_ok=True)
with open("results/seed_study.json", "w") as f:
    json.dump({"config": "minmax + overlap", "split_seed": 12, "training_seeds": N_SEEDS,
               "runs": runs}, f, indent=2)

In [ ]:
!python scripts/make_seed_figure.py
from IPython.display import Image, display
display(Image("assets/seed_study.png"))

In [ ]:
!zip -q seed_study_outputs.zip results/seed_study.json assets/seed_study.png
try:
    from google.colab import files
    files.download("seed_study_outputs.zip")
except ImportError:
    print("wrote seed_study_outputs.zip")